# Local DPO pilot
Independent local copy of dpo_train.ipynb. Uses DL_Final, the local Qwen weights and MedQA files, 64 preference pairs, 32 validation questions, and 10 optimizer steps. Original-model baseline evaluation is disabled. Outputs use a separate local directory. No full-data training is started.

In [ ]:
from pathlib import Path
import sys, os, json, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = Path('/content/drive/MyDrive/medqa-calibration')
else:
    here = Path.cwd().resolve()
    REPO = here if (here / 'train/dpo/train.py').is_file() else here.parent
assert (REPO / 'train/dpo/train.py').is_file(), 'Set REPO to the repository root.'
print('Repository:', REPO)

In [ ]:
import torch, importlib.metadata
assert torch.cuda.is_available(), 'CUDA is required.'
print('Python:', sys.executable)
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('BF16 supported:', torch.cuda.is_bf16_supported())
assert 'DL_Final' in sys.executable, 'Select the DL_Final kernel.'


## Dependencies
Activate `DL_Final` before opening this notebook locally. Retain Colab's installed CUDA-enabled PyTorch. Set `INSTALL_DEPENDENCIES=True` only when the DPO dependencies need installation; restart the kernel afterward if the notebook asks for it.

In [ ]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'train/dpo/requirements-local.txt')])
import importlib.metadata
for name in ['torch', 'transformers', 'trl', 'peft', 'datasets', 'accelerate', 'bitsandbytes']:
    try:
        print(name, importlib.metadata.version(name))
    except importlib.metadata.PackageNotFoundError:
        print(name, 'not installed')

In [ ]:
def run_command(command):
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
                               env=dict(os.environ, PYTHONIOENCODING='utf-8', PYTHONUNBUFFERED='1'))
    log_dir = REPO / 'outputs/dpo_local_pilot_execution'
    log_dir.mkdir(parents=True, exist_ok=True)
    try:
        with (log_dir / 'commands.log').open('a', encoding='utf-8') as log:
            log.write('Command: ' + repr(command) + '\n')
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
        if process.wait() != 0:
            raise RuntimeError('Command failed; inspect the output above.')
    except BaseException:
        if process.poll() is None:
            process.terminate()
            process.wait()
        raise
    finally:
        process.stdout.close()


## One-time original-model baseline
Run this optional stage before the first training experiment. It evaluates all 500 fixed validation questions and the full official MedQA test split, without adapters or answer generation. Subsequent experiments can leave `RUN_BASELINE=False` and reuse the saved results. Use the same base revision, quantization, and precision for comparable results; if these change, use a separate baseline output directory. Existing outputs are protected from overwrite. A failed partial run requires a new output directory. Test scores must never guide hyperparameter or checkpoint selection.


In [ ]:
RUN_BASELINE = False  # Enable for the first run only; later runs display saved results.
BASELINE_SOURCE = 'local'
BASELINE_MODEL = 'Qwen/Qwen2.5-3B-Instruct' if BASELINE_SOURCE == 'hf' else str(REPO.parent / 'Qwen2.5-3B-Instruct')
BASELINE_QUANTIZATION = 'none'
BASELINE_COMPUTE_DTYPE = 'bf16'
BASELINE_BATCH_SIZE = 4
BASELINE_OUTPUT = REPO / 'outputs/base_medqa_bf16'


In [ ]:
if RUN_BASELINE:
    command = [sys.executable, '-u', '-m', 'train.dpo.baseline',
               '--source', BASELINE_SOURCE, '--model', BASELINE_MODEL,
               '--quantization', BASELINE_QUANTIZATION, '--compute_dtype', BASELINE_COMPUTE_DTYPE,
               '--batch_size', str(BASELINE_BATCH_SIZE), '--output_dir', str(BASELINE_OUTPUT)]
    if BASELINE_SOURCE == 'local':
        command += ['--local_train_file', str(REPO.parent / 'MedQA-USMLE-4-options/phrases_no_exclude_train.jsonl'),
                    '--local_test_file', str(REPO.parent / 'MedQA-USMLE-4-options/phrases_no_exclude_test.jsonl')]
    run_command(command)
else:
    print('Baseline evaluation disabled; existing results will be displayed if available.')


In [ ]:
import pandas as pd
from IPython.display import display
if (BASELINE_OUTPUT / 'summary.json').is_file():
    baseline = json.loads((BASELINE_OUTPUT / 'summary.json').read_text())
    manifest = json.loads((BASELINE_OUTPUT / 'manifest.json').read_text())
    display(pd.DataFrame([{'split': name, 'questions': manifest['counts'][name], **metrics}
                          for name, metrics in baseline.items()]))
    print('Saved baseline configuration:', manifest['model'], manifest['quantization'], manifest['compute_dtype'])
    print('Test scores are descriptive only and must not be used for hyperparameter selection.')
else:
    print('No completed baseline exists. Set RUN_BASELINE=True to evaluate the original model.')


## Run configuration
`PILOT=True` uses 64 pairs, 32 validation questions, and 10 updates in a separate directory. Set it to False for full training. Changing a pilot into a full run requires a new output directory. Use `STOP_AFTER_STEP` instead when pausing a full run while preserving its learning-rate schedule.

Keep the output directory on Google Drive in Colab. Resume requires an existing complete checkpoint and matching arguments.

In [ ]:
SOURCE = 'local'
MODEL = 'Qwen/Qwen2.5-3B-Instruct' if SOURCE == 'hf' else str(REPO.parent / 'Qwen2.5-3B-Instruct')
LOCAL_TRAIN_FILE = REPO.parent / 'MedQA-USMLE-4-options/phrases_no_exclude_train.jsonl'
DATA_FILE = REPO / 'outputs/medqa5000_L4_seed42_qpb2/selection_seed42/dpo.jsonl'
PILOT = True
RUN_TRAINING = True
RESUME = False
SEED = 1
QUANTIZATION = 'none'
COMPUTE_DTYPE = 'bf16'
EPOCHS = 1
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 16
MAX_LENGTH = 2048
LEARNING_RATE = 5e-6
BETA = 0.1
LORA_DROPOUT = 0.05
STOP_AFTER_STEP = 0
OUTPUT = REPO / 'outputs' / 'dpo_local_pilot_bf16_lora_seed1'
COMMON = [sys.executable, '-u', '-m', 'train.dpo.train', '--source', SOURCE,
          '--model', MODEL, '--data_file', str(DATA_FILE), '--output_dir', str(OUTPUT),
          '--seed', str(SEED), '--quantization', QUANTIZATION, '--compute_dtype', COMPUTE_DTYPE, '--num_train_epochs', str(EPOCHS),
          '--batch_size', str(BATCH_SIZE), '--gradient_accumulation_steps', str(GRADIENT_ACCUMULATION),
          '--max_length', str(MAX_LENGTH), '--learning_rate', str(LEARNING_RATE), '--beta', str(BETA),
          '--lora_dropout', str(LORA_DROPOUT), '--no-baseline_validation', '--eval_batch_size', '2', '--logging_steps', '1']
if SOURCE == 'local':
    COMMON += ['--local_train_file', str(LOCAL_TRAIN_FILE)]
if PILOT:
    COMMON += ['--n_pairs', '64', '--n_val', '32', '--max_steps', '10']
if STOP_AFTER_STEP:
    COMMON += ['--stop_after_step', str(STOP_AFTER_STEP)]
print('Output:', OUTPUT)
print('Training enabled:', RUN_TRAINING)

## Data and token-length check
This checks source hashes, question identities, held-out separation, pair labels, and chat-template lengths. It loads the tokenizer, not model weights. Archived raw samples are not needed.

In [ ]:
run_command(COMMON + ['--check_data'])


## Train or resume
Default: unquantized BF16 LoRA, rank 16, alpha 32, active LoRA dropout 0.05, and a fixed base reference. No new responses are sampled. The original-model baseline is handled by the optional stage above and is not repeated here. Adapter validation runs after each epoch. Best-adapter selection uses validation accuracy, then NLL. Test metrics never select a checkpoint. Training checkpoints include optimizer and RNG state; `best_adapter` is for evaluation, not resume.


In [ ]:
if RUN_TRAINING:
    run_command(COMMON + (['--resume'] if RESUME else []))
else:
    print('Training is disabled. Review the configuration and set RUN_TRAINING=True to start.')

## Results
Review training loss separately from medical answer accuracy. ECE, Brier, NLL, and high-confidence errors measure confidence reliability. Timing estimates exclude validation and checkpoint overhead.

In [ ]:
import pandas as pd
from IPython.display import display
summary_file = OUTPUT / 'training_summary.json'
if summary_file.is_file():
    summary = json.loads(summary_file.read_text())
    display(pd.DataFrame([summary['best_validation']['metrics']]))
    print(json.dumps({k: summary[k] for k in ['global_step', 'epoch', 'stopped_early',
          'steady_step_seconds', 'estimated_training_seconds_per_epoch', 'peak_allocated_gib']}, indent=2))
    print('Best adapter:', OUTPUT / 'best_adapter')
else:
    print('No completed training summary exists for this output directory.')
log = OUTPUT / 'training_metrics.jsonl'
if log.is_file():
    history = pd.read_json(log, lines=True)
    display(history.tail(10))
    if 'loss' in history:
        import matplotlib.pyplot as plt
        history.dropna(subset=['loss']).plot(x='step', y='loss', title='DPO training loss', legend=False)
        plt.ylabel('Loss')
        plt.show()

## Final experiments
After validation-based configuration selection, repeat with a second seed and a separate output directory. Evaluate selected adapters on the MedQA test set and transfer datasets using the existing evaluator. Do not use those scores to choose training checkpoints. See `train/dpo/README.txt` for commands and artifact descriptions.

## Reopening this completed pilot
The saved outputs document the completed 10-step run. To refresh the report without retraining, run `python -m train.dpo.run_local_training --report-only` from the repository root with DL_Final active. For another pilot, select a new output directory. The max_steps=10 setting overrides the one-epoch setting and covers 2.5 passes over these 64 pairs.

## BF16 LoRA configuration
The base model is loaded without quantization in BF16 and remains frozen. Only LoRA parameters are trained; optimizer updates retain FP32 adapter parameters. BF16 support is required, with no automatic FP16 fallback. This run uses a separate output directory and must not resume a QLoRA checkpoint. For subsequent evaluation, use `--quantization none --compute_dtype bf16`. Previous executed notebooks are preserved in the sibling backup directory.
